In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from scipy import interpolate
from scipy.interpolate import UnivariateSpline
from scipy.interpolate import make_interp_spline
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

import utilities.plot_settings

In [ ]:
def failure_rate_fit(
    rate_initial: float,
    t: np.ndarray,
    tau: float,
    index: float,
) -> np.ndarray:
    """
    An analytical function to fit the failure rate as a function of time.

    Args:
        rate_initial (float): Initial failure rate as number of events per year.
        t (np.ndarray): Time in [yr].
        tau (float): Timescale in [yr] when to start the power-law decay.
        index (float): Power-law index for the decay in failure rate in time.

    Returns:
        (np.ndarray): failure rate as a function of time t.
    """

    rate = rate_initial * (1 + t / tau) ** index

    return rate

In [ ]:
# Read the file as space-delimited.
df_1e14_H = pd.read_csv('../../failures/failures_50-50_1e14_H.d', delim_whitespace=True, header=None)
df_1e15_H = pd.read_csv('../../failures/failures_50-50_1e15_H.d', delim_whitespace=True, header=None)
df_5e15_H = pd.read_csv('../../failures/failures_50-50_5e15_H.d', delim_whitespace=True, header=None)

# Assign column names.
df_1e14_H.columns = ['time', 'energy', 'theta', 'radius', 'volume', 'timescale']
df_1e15_H.columns = ['time', 'energy', 'theta', 'radius', 'volume', 'timescale']
df_5e15_H.columns = ['time', 'energy', 'theta', 'radius', 'volume', 'timescale']

In [ ]:
t_1e14_H = np.array(df_1e14_H['time'].values)
t_1e15_H = np.array(df_1e15_H['time'].values)
t_5e15_H = np.array(df_5e15_H['time'].values)

E_1e14_H = np.array(df_1e14_H['energy'].values)
E_1e15_H = np.array(df_1e15_H['energy'].values)
E_5e15_H = np.array(df_5e15_H['energy'].values)

In [ ]:
mask_E_1e14_H = E_1e14_H > 1.e41
mask_E_1e15_H = E_1e15_H > 1.e41
mask_E_5e15_H = E_5e15_H > 1.e41

In [ ]:

t_1e14_H = t_1e14_H[mask_E_1e14_H]
t_1e15_H = t_1e15_H[mask_E_1e15_H]
t_5e15_H = t_5e15_H[mask_E_5e15_H]

E_1e14_H = E_1e14_H[mask_E_1e14_H]
E_1e15_H = E_1e15_H[mask_E_1e15_H]
E_5e15_H = E_5e15_H[mask_E_5e15_H]


In [ ]:
# Define an array with the log10 of the initial magnetic field values for the different cooling curves.
log_B0 = np.array([14, 15, np.log10(5.0e15)])

# Define initial magnetic fields where to evaluate the interpolated failure rate curves.
# Note that this is needed now to set the right colors.
log_B0_eval = np.linspace(11.0, 16.0, 100)

# Combine the arrays to find the global min and max values.
combined_values = np.concatenate([log_B0, log_B0_eval])
vmin, vmax = combined_values.min(), combined_values.max()

# Create a colormap and normalize it.
cmap = plt.cm.viridis
norm = Normalize(vmin=vmin, vmax=vmax)

In [ ]:
# Define bin edges
#t_edges = np.logspace(2.0, 6.0, 1001)
t_edges = np.linspace(0.0, 1.e6, 10001)
bin_widths = np.diff(t_edges)
bin_centers = t_edges[:-1] + bin_widths / 2

# Compute the histogram
counts_1e14_H, _ = np.histogram(t_1e14_H, bins=t_edges)
counts_1e15_H, _ = np.histogram(t_1e15_H, bins=t_edges)
counts_5e15_H, _ = np.histogram(t_5e15_H, bins=t_edges)

# Normalize by bin width
rate_1e14_H = counts_1e14_H / bin_widths 
rate_1e15_H = counts_1e15_H / bin_widths 
rate_5e15_H = counts_5e15_H / bin_widths

# Plot using ax.step
fig, ax = plt.subplots(figsize=(15, 8))
ax.step(
    bin_centers,
    rate_1e14_H,
    where='post',
    color=cmap(norm(log_B0[0])),
    rasterized=True,
    lw=4,
    alpha=1,
)
ax.step(
    bin_centers,
    rate_1e15_H,
    where='post',
    color=cmap(norm(log_B0[1])),
    rasterized=True,
    lw=4,
    alpha=1,
)
ax.step(
    bin_centers,
    rate_5e15_H,
    where='post',
    color=cmap(norm(log_B0[2])),
    rasterized=True,
    lw=4,
    alpha=1,
)

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

ax.set_xlabel("time [yr]")
ax.set_ylabel("Failures per year")
ax.set_xscale("log")
ax.set_yscale("log")

In [ ]:
time_grid = np.logspace(0., 7., 50)

failure_rate_fit_1e14 = failure_rate_fit(3.e-1, time_grid, 2.e2, -2)
failure_rate_fit_1e15 = failure_rate_fit(1.e1, time_grid, 1.e3, -2)
failure_rate_fit_5e15 = failure_rate_fit(1.e2, time_grid, 2.e3, -2)

In [ ]:
fr1e14_H_interpolator = interpolate.InterpolatedUnivariateSpline(
    bin_centers, rate_1e14_H, k=1
)
fr1e15_H_interpolator = interpolate.InterpolatedUnivariateSpline(
    bin_centers, rate_1e15_H, k=1
)
fr5e15_H_interpolator = interpolate.InterpolatedUnivariateSpline(
    bin_centers, rate_5e15_H, k=1
)

rate_1e14_H_smooth = fr1e14_H_interpolator(time_grid)
rate_1e15_H_smooth = fr1e15_H_interpolator(time_grid)
rate_5e15_H_smooth = fr5e15_H_interpolator(time_grid)

# Plot using ax.step
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    time_grid,
    rate_1e14_H_smooth,
    lw=4,
    alpha=1,
    color=cmap(norm(log_B0[0])),
    rasterized=True,
    label=r"1e14 H"
)
ax.plot(
    time_grid,
    rate_1e15_H_smooth,
    color=cmap(norm(log_B0[1])),
    rasterized=True,
    lw=4,
    alpha=1,
    label=r"1e15 H"
)
ax.plot(
    time_grid,
    rate_5e15_H_smooth,
    color=cmap(norm(log_B0[2])),
    rasterized=True,
    lw=4,
    alpha=1,
    label=r"5e15 H"
)

ax.plot(
    time_grid,
    failure_rate_fit_1e14,
    lw=4,
    ls=":",
    alpha=1,
    color=cmap(norm(log_B0[0])),
    rasterized=True,
    label=r"1e14 H"
)
ax.plot(
    time_grid,
    failure_rate_fit_1e15,
    color=cmap(norm(log_B0[1])),
    rasterized=True,
    lw=4,
    ls=":",
    alpha=1,
    label=r"1e15 H"
)
ax.plot(
    time_grid,
    failure_rate_fit_5e15,
    color=cmap(norm(log_B0[2])),
    rasterized=True,
    lw=4,
    ls=":",
    alpha=1,
    label=r"5e15 H"
)

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

ax.set_xlabel("time [yr]")
ax.set_ylabel("Failures per year")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(0.0, 1000000)
ax.set_ylim(0.0001, 200)

In [ ]:
# Create a grid of initial magnetic fields and stack together all the cooling curves.
B0 = 10**log_B0
rate_t_stack = np.vstack(
    (failure_rate_fit_1e14, failure_rate_fit_1e15, failure_rate_fit_5e15)
).T

# Define the minimum and maximum age in [yr] and the minimum and maximum initial magnetic field in [G].
time_range = np.array([0.0, 1.0e7])
B0_range = np.array([1.0e11, 1.0e17])

failure_rate_interpolator = interpolate.RectBivariateSpline(
    time_grid,
    B0,
    rate_t_stack,
    bbox=[time_range[0], time_range[1], B0_range[0], B0_range[1]],
    kx=1,
    ky=1,
)

In [ ]:
# Save the interpolator function and try to import it again to see if it works.
interpolator_path = "interpolator_crust_failure_rate.pkl"
with open(
    interpolator_path,
    "wb",
) as f:
    pickle.dump(failure_rate_interpolator, f)

with open(
    interpolator_path,
    "rb",
) as f:
    failure_rate_interpolator_import = pickle.load(f)

In [ ]:
# Define a grid of times and initial magnetic fields at which we evaluate the interpolated cooling curves.
t_eval = np.logspace(0.0, 7, 500)
B0_eval = 10**log_B0_eval

failure_rate_interp = failure_rate_interpolator_import(t_eval, B0_eval)
print(failure_rate_interp.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(1.0e-5, 1.0e3)
ax.set_xlabel(r"Time [yr]")
ax.set_ylabel("Failures per year")

ax.plot(
    time_grid,
    rate_1e14_H_smooth,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[0])),
    rasterized=True,
)
ax.plot(
    time_grid,
    rate_1e15_H_smooth,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[1])),
    rasterized=True,
)
ax.plot(
    time_grid,
    rate_5e15_H_smooth,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[2])),
    rasterized=True,
)

for i in range(len(B0_eval)):
    ax.plot(
        t_eval,
        failure_rate_interp[:, i],
        linestyle="--",
        linewidth=4,
        color=cmap(norm(log_B0_eval[i])),
        rasterized=True,
        alpha=0.5,
    )

# Create a legend for the line styles.
style_handles = [
    Line2D(
        [0], [0], color="black", linestyle="-", linewidth=4, label="Original"
    ),
    Line2D(
        [0],
        [0],
        color="black",
        linestyle="--",
        linewidth=4,
        label="Interpolated",
    ),
]
plt.legend(handles=style_handles, frameon=False, loc=0)

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

plt.grid()

In [ ]:
# Try to evaluate the failure rate for a set of random values of the age and the initial magnetic field.
t_eval_test = np.array([1.0e3, 1.0e2])
B0_eval_test = np.logspace(13.0, 15, 2)

failure_rate_interp_test = failure_rate_interpolator_import.ev(t_eval_test, B0_eval_test)
print(failure_rate_interp_test)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    t_1e14_H,
    E_1e14_H,
    linestyle="None",
    marker="o",
    color=cmap(norm(log_B0[0])),
    markersize=6,
    alpha=0.1,
    rasterized=True,
)
ax.plot(
    t_1e15_H,
    E_1e15_H,
    linestyle="None",
    marker="o",
    color=cmap(norm(log_B0[1])),
    markersize=6,
    alpha=0.1,
    rasterized=True,
)

ax.plot(
    t_5e15_H,
    E_5e15_H,
    linestyle="None",
    marker="o",
    color=cmap(norm(log_B0[2])),
    markersize=6,
    alpha=0.1,
    rasterized=True,
)

ax.set_xlabel("Time [yr]")
ax.set_ylabel("Crustal failure energy [erg]")
ax.set_xscale("log")
ax.set_yscale("log")

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

plt.grid()

## 